In [ ]:
!pip install -qqqq -U langchain-openai
!pip install -qqqq -U langchain-core
!pip install -qqqq -U openai
!pip install -qqqq -U pydantic
!pip install -qqqq -U langgraph

In [ ]:
import os
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt

# Import the Python SDK for Colab secrets
from google.colab import userdata
from IPython.display import display,Image

In [ ]:
import os

# Fetch the API key from Colab secrets
openai_api_key = userdata.get('OPENAI_APIKEY')

# Set up Environmental Variables
os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
# ==========================================
# 1. DEFINE TOOLS WITH THE @tool DECORATOR
# ==========================================

@tool
def write_file(filename: str, content: str) -> str:
    """Write content to a file. Requires human approval before execution."""
    # Simulating file write operation
    return f"Successfully wrote content to {filename}"

@tool
def read_file(filename: str) -> str:
    """Read contents of a file. Safe to run automatically without review."""
    return f"Contents of {filename}: [Sample file data line 1, line 2]"

tools_map = {
    "write_file": write_file,
    "read_file": read_file
}


In [ ]:
# ==========================================
# 2. BIND TOOLS TO CHATOPENAI
# ==========================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(list(tools_map.values()))

In [ ]:
# ==========================================
# 2. DEFINE GRAPH STATE
# ==========================================

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# ==========================================
# 3. DEFINE GRAPH NODE
# ==========================================

def hitl_call_llm(state: AgentState):
    """LLM Agent node that decides whether to respond or call tools."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> Literal["tools", END]:
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

def hitl_tool_node(state: AgentState):
    """Executes tools, intercepting sensitive actions with an interrupt."""
    messages = state["messages"]
    print(f"\nmessages: {messages}\n")
    last_message = messages[-1]

    tool_results = []

    # Step B: Check if the model triggered any tool calls
    if last_message:
        for tool_call in last_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]
            print(f"\n--- Tool Call: {tool_name} ---")
            print(f"Arguments: {tool_args}")
            # Sensitive tools require HITL interruption
            if tool_name == "write_file":
                print(f"\n--- 🛑 HITL PAUSE: Sensitive Action Requested ---")
                print(f"Tool: {tool_name}")
                print(f"Arguments: {tool_args}")

                # Pass payload to the client/runtime
                decision = interrupt({
                "action": tool_name,
                "arguments": tool_args,
                "message": f"Approve writing content to '{tool_args.get('filename')}'?"
                })

                # The value passed via Command(resume=...) comes back as 'decision'
                print(f"--- ▶️ RESUMED WITH DECISION: {decision} ---")
                if decision == 'y':
                    print(f"--- [Approved] Executing {tool_name} ---")
                    tool_result = tools_map[tool_name].invoke(tool_args)
                    tool_results.append(ToolMessage(content=tool_result, tool_call_id=tool_call_id))
                elif decision == 'n':
                    print(f"--- [Rejected] Action denied by human ---")
                    # messages[0].content = "You have rejected the execution"
                    tool_results[0] = HumanMessage(content="You have rejected the execution")
                    tool_results.append(ToolMessage(content=messages[0].content, tool_call_id=tool_call_id))
                else:
                   print(f"--- [Wrong] Choice by human ---")
                   rejection_msg = "Error: Wrong choice send ."
                   tool_results[0].content = "You should have choosen either y or n "
                   tool_results.append(ToolMessage(content=messages[0].content, tool_call_id=tool_call_id))
            elif tool_name == "read_file":
              # Safe tools execute automatically without pausing
              print(f"--- [Auto-Executing Safe Tool] {tool_name} ---")
              tool_result = tools_map[tool_name].invoke(tool_args)
              tool_results.append(ToolMessage(content=tool_result, tool_call_id=tool_call_id))
            else :
              print(f"--- [Unhandled Tool Call] {tool_name} ---")

        return {"messages": tool_results}
    else:
        return {"messages": []}

In [ ]:
# ==========================================
# 3. BUILD THE GRAPH WITH A CHECKPOINTER
# ==========================================

# Define Nodes
builder = StateGraph(AgentState)
builder.add_node("agent", hitl_call_llm)
builder.add_node("tools", hitl_tool_node)

# Define Edges
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue)
builder.add_edge("tools", "agent")

# A checkpointer is required for interrupt() to save and resume graph state
memory = MemorySaver()

graph = builder.compile(checkpointer=memory)



In [ ]:
# ============================================================================
# Show Chatbot Graph
# ============================================================================

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# ==========================================
# 3. EXECUTION
# ==========================================

# This prompt will trigger 'write_file', which activates the HITL prompt pause

thread_config = {"configurable": {"thread_id": "hitl_thread_1"}}
initial_input = {"messages": [HumanMessage(content="Write a welcome note to summary.txt")]}

print("--- 🚀 Invoking Graph Run ---")
graph.invoke(initial_input, thread_config)


# Check if the graph has paused at an interrupt
state = graph.get_state(thread_config)

if state.tasks and state.tasks[0].interrupts:
  # Extract the payload sent by interrupt()
    interrupt_payload = state.tasks[0].interrupts[0].value
    print(f"\n[Pending Approval Request]: {interrupt_payload['message']}")
    print(f"Arguments: {interrupt_payload['arguments']}")

    # Simulate human decision via input prompt
    user_choice = input("Approve action? (y/n): ").strip().lower()
    print("\n--- 🔄 Resuming Graph Execution with Command ---")
    graph.invoke(Command(resume=user_choice), thread_config)


# Print final output from the completed graph state
final_state = graph.get_state(thread_config)
print("\n==============================")
print("FINAL AGENT RESPONSE:")
print("==============================\n")
print(final_state.values["messages"][-1].content)
